In [1]:
import pandas as pd 

data1 = pd.read_feather(rf'/project/model_share/share_1_new/factor_csy/fac20250625/fac20250625.fea')
data2 = pd.read_feather(rf'/project/model_share/share_1_new/factor_csy/fac20250715/fac20250715.fea')


In [2]:
data1.info

<bound method DataFrame.info of              date    Code   fac0001   fac0002   fac0003   fac0004   fac0005  \
0        20210104  000001  0.103297  0.508136 -0.277977 -4.407612  0.259874   
1        20210104  000002  0.041108  0.692095 -0.148797 -3.662460  0.388681   
2        20210104  000004  0.061697  0.900875 -0.046951 -1.091614  0.458014   
3        20210104  000005  0.000000  0.932106 -0.026149  0.495133  0.558257   
4        20210104  000006  0.032552  0.958341 -0.025876 -1.673883  0.331534   
...           ...     ...       ...       ...       ...       ...       ...   
4497842  20241231  688799  0.427302       NaN       NaN  0.868741       NaN   
4497843  20241231  688800  0.509672  0.568677 -0.055477  1.321744  0.674025   
4497844  20241231  688819  0.020425  0.971780 -0.017278 -0.562102  0.300689   
4497845  20241231  688981  0.103586  0.720491 -0.165714 -2.509371  0.283653   
4497846  20241231  689009  0.136386  0.884088 -0.031163  1.298170  0.617184   

          fac0006  

In [3]:
data2.info

<bound method DataFrame.info of              date    Code  a0001_factor_fea_20250701  \
0        20210104  000001                   5.799058   
1        20210104  000002                   4.615568   
2        20210104  000004                   4.374511   
3        20210104  000005                   3.952368   
4        20210104  000006                   4.769330   
...           ...     ...                        ...   
5441899  20241231  873703                        NaN   
5441900  20241231  873706                        NaN   
5441901  20241231  873726                        NaN   
5441902  20241231  873806                        NaN   
5441903  20241231  873833                        NaN   

         a0002_factor_fea_20250701  a0003_factor_fea_20250701  \
0                         5.312016                   3.198957   
1                         3.648163                   4.446108   
2                         3.401806                   4.862444   
3                         3.205059 

In [4]:
import numpy as np
def normed(df):
    df_num = df.select_dtypes(include=[np.number])
    df_num = np.clip(df_num, df_num.quantile(0.005), df_num.quantile(0.995), axis=1)
    df_num = (df_num - df_num.mean()) / df_num.std()
    df_num = df_num.fillna(0)
    # 将数值列替换回原始 DataFrame（保留非数值列）
    df[df_num.columns] = df_num
    return df
data1 = normed(data1)
data2 = normed(data2)

In [5]:
data1.info

<bound method DataFrame.info of              date    Code   fac0001   fac0002   fac0003   fac0004   fac0005  \
0        20210104  000001 -0.264369 -1.964814 -1.791665 -1.959326 -0.530822   
1        20210104  000002 -0.707439 -0.797542 -0.618829 -1.536962 -0.254150   
2        20210104  000004 -0.560753  0.527224  0.305837 -0.079767 -0.105226   
3        20210104  000005 -1.000310  0.725396  0.494699  0.819626  0.110092   
4        20210104  000006 -0.768393  0.891863  0.497176 -0.409806 -0.376900   
...           ...     ...       ...       ...       ...       ...       ...   
4497842  20241231  688799  2.043995  0.000000  0.000000  1.031393  0.000000   
4497843  20241231  688800  2.630835 -1.580665  0.228432  1.288162  0.358759   
4497844  20241231  688819 -0.854792  0.977139  0.575243  0.220369 -0.443153   
4497845  20241231  688981 -0.262313 -0.617361 -0.772420 -0.883373 -0.479745   
4497846  20241231  689009 -0.028633  0.420709  0.449180  1.274800  0.236664   

          fac0006  

In [6]:
data2.info

<bound method DataFrame.info of              date    Code  a0001_factor_fea_20250701  \
0        20210104  000001                   0.703864   
1        20210104  000002                  -0.439346   
2        20210104  000004                  -0.672198   
3        20210104  000005                  -1.079973   
4        20210104  000006                  -0.290817   
...           ...     ...                        ...   
5441899  20241231  873703                   0.000000   
5441900  20241231  873706                   0.000000   
5441901  20241231  873726                   0.000000   
5441902  20241231  873806                   0.000000   
5441903  20241231  873833                   0.000000   

         a0002_factor_fea_20250701  a0003_factor_fea_20250701  \
0                         0.711113                  -1.210107   
1                        -0.743374                  -0.031650   
2                        -0.958731                   0.361754   
3                        -1.130721 

In [7]:

# 合并数据（横向拼接）
# 2. 设置索引并排序（加速合并）
data1 = data1.set_index(['date', 'Code']).sort_index()
data2 = data2.set_index(['date', 'Code']).sort_index()



In [ ]:
# 3. 合并数据（使用 merge 替代 join，更稳定且效率高）
merged_data = pd.merge(
    data1, data2,
    how='left',
    left_index=True,
    right_index=True,
    sort=False  # 避免不必要排序开销
)

merged_data = merged_data.reset_index()
merged_data.to_feather('/project/model_share/share_1_new/factor_csy/fac20250715_all/merged_data.fea')

ValueError: do not recognize join method letf